# Gradient Boosting Regressor — Food Delivery Time Prediction

This notebook adds a Gradient Boosting Regressor to the delivery time models. It follows the same 80/20 train/test split and reports MAE, MSE, RMSE, and R². Hyperparameters are selected by cross-validation on the training split; the test split is used for the final evaluation.

The target is `Time_taken_min`. `Average_Speed_kmph` is excluded because it can reveal information from the completed delivery. Order identifiers and the raw date are excluded as well.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

RANDOM_STATE = 42
DATA_PATH = Path(r'D:\innomatics\Machine Learning\food delivary time prediction\Food_Delivery_Time_Prediction.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('Food_Delivery_Time_Prediction.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place Food_Delivery_Time_Prediction.csv beside this notebook or update DATA_PATH.')

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
display(df.head())

In [ ]:
target = 'Time_taken_min'
excluded = [col for col in ['Average_Speed_kmph', 'Order_ID', 'Order_Date'] if col in df.columns]
if target not in df.columns:
    raise KeyError(f'Required target column {target!r} is missing.')

X = df.drop(columns=[target, *excluded]).copy()
y = pd.to_numeric(df[target], errors='coerce')
valid_rows = y.notna() & np.isfinite(y)
X, y = X.loc[valid_rows].copy(), y.loc[valid_rows].astype(float)
print('Excluded columns:', excluded)
print('Feature columns:', X.columns.tolist())
print('Rows available:', len(X))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

ordinal_categories = {
    'Restaurant_Load': ['Low', 'Medium', 'High'],
    'Delivery_Distance_Category': ['Short', 'Medium', 'Long'],
    'Traffic_Level': ['Low', 'Moderate', 'High', 'Severe'],
    'Delivery_Priority': ['Normal', 'Priority', 'VIP'],
}
ordinal_cols = [col for col in ordinal_categories if col in X_train.columns]
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
nominal_cols = [col for col in X_train.columns if col not in numeric_cols + ordinal_cols]
print('Numerical:', numeric_cols)
print('Ordinal:', ordinal_cols)
print('Nominal:', nominal_cols)

In [ ]:
transformers = []
if numeric_cols:
    transformers.append(('numeric', SimpleImputer(strategy='median'), numeric_cols))
if nominal_cols:
    nominal_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    transformers.append(('nominal', nominal_pipe, nominal_cols))
if ordinal_cols:
    ordinal_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            categories=[ordinal_categories[col] for col in ordinal_cols],
            handle_unknown='use_encoded_value', unknown_value=-1
        )),
    ])
    transformers.append(('ordinal', ordinal_pipe, ordinal_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))
])
gb_pipeline

## Baseline Gradient Boosting

Gradient Boosting builds trees in sequence, with each new tree correcting errors made by the previous ones. Unlike KNN, it does not need feature scaling. One-hot encoding is set to produce a dense matrix, as this estimator expects dense input.

In [ ]:
baseline_model = gb_pipeline.set_params(
    model__n_estimators=150, model__learning_rate=0.05, model__max_depth=2
)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R²': r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_test, y_pred_baseline)
print('Baseline Gradient Boosting results')
pd.Series(baseline_metrics).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred_baseline, alpha=0.35)
limits = [min(y_test.min(), y_pred_baseline.min()), max(y_test.max(), y_pred_baseline.max())]
axes[0].plot(limits, limits, '--', color='red')
axes[0].set(xlabel='Actual delivery time (min)', ylabel='Predicted delivery time (min)', title='Baseline: actual vs predicted')
axes[1].scatter(y_pred_baseline, y_test - y_pred_baseline, alpha=0.35)
axes[1].axhline(0, linestyle='--', color='red')
axes[1].set(xlabel='Predicted delivery time (min)', ylabel='Residual (actual - predicted)', title='Baseline residuals')
plt.tight_layout()
plt.show()

## Tune the model

Randomized search selects parameters by 3-fold cross-validated MAE on the training data. This keeps the search practical on the 50,000-row dataset. The held-out test set is not used during tuning.

In [ ]:
param_distributions = {
    'model__n_estimators': [100, 150, 250],
    'model__learning_rate': [0.03, 0.05, 0.08, 0.1],
    'model__max_depth': [1, 2, 3],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__subsample': [0.7, 0.85, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)
random_search.fit(X_train, y_train)
print('Best parameters:', random_search.best_params_)
print(f'Best 3-fold CV MAE: {-random_search.best_score_:.3f} min')

In [ ]:
best_gb = random_search.best_estimator_
y_pred_tuned = best_gb.predict(X_test)
tuned_metrics = regression_metrics(y_test, y_pred_tuned)
comparison = pd.DataFrame(
    [baseline_metrics, tuned_metrics],
    index=['Baseline Gradient Boosting', 'Tuned Gradient Boosting']
)
display(comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred_tuned, alpha=0.35)
limits = [min(y_test.min(), y_pred_tuned.min()), max(y_test.max(), y_pred_tuned.max())]
axes[0].plot(limits, limits, '--', color='red')
axes[0].set(xlabel='Actual delivery time (min)', ylabel='Predicted delivery time (min)', title='Tuned: actual vs predicted')
axes[1].scatter(y_pred_tuned, y_test - y_pred_tuned, alpha=0.35)
axes[1].axhline(0, linestyle='--', color='red')
axes[1].set(xlabel='Predicted delivery time (min)', ylabel='Residual (actual - predicted)', title='Tuned residuals')
plt.tight_layout()
plt.show()

## Feature importance and model comparison

Impurity-based feature importance gives a quick indication of which transformed inputs were most useful. It can favor categories with many possible values, so treat it as an exploratory view. Compare the test MAE and RMSE below with the other model notebooks using the same split and features. Lower MAE and RMSE are better; higher R² is better.

In [ ]:
feature_names = best_gb.named_steps['preprocessor'].get_feature_names_out()
importances = best_gb.named_steps['model'].feature_importances_
feature_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
display(feature_importance.head(20).to_frame('Importance'))
feature_importance.head(20).sort_values().plot(kind='barh', figsize=(9, 7), title='Top Gradient Boosting feature importances')
plt.xlabel('Impurity-based importance')
plt.tight_layout()
plt.show()